In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Bounding boxes for YOLO classification model


In [3]:
import pandas as pd
path = '/content/drive/MyDrive/Colab Notebooks/DF20M-metadata/'
test_df = pd.read_csv(path + 'DF20M-public_test_metadata_PROD.csv')

In [4]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3640 entries, 0 to 3639
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   gbifID                3640 non-null   int64  
 1   eventDate             3639 non-null   object 
 2   year                  3639 non-null   float64
 3   month                 3639 non-null   float64
 4   day                   3639 non-null   float64
 5   countryCode           3640 non-null   object 
 6   locality              3636 non-null   object 
 7   taxonID               3640 non-null   float64
 8   scientificName        3640 non-null   object 
 9   kingdom               3640 non-null   object 
 10  phylum                3640 non-null   object 
 11  class                 3640 non-null   object 
 12  order                 3640 non-null   object 
 13  family                3640 non-null   object 
 14  genus                 3640 non-null   object 
 15  specificEpithet      

In [5]:
test_df.head()

,gbifID,eventDate,year,month,day,countryCode,locality,taxonID,scientificName,kingdom,...,level2Gid,level2Name,ImageUniqueID,Substrate,rightsHolder,Latitude,Longitude,CoorUncert,Habitat,image_path
0,2238504270,2017-08-20T00:00:00,2017.0,8.0,20.0,DK,Store Dyrehave,20020.0,Russula nigricans (Bull.) Fr.,Fungi,...,DNK.1.20_1,Hillerød,2238504270-171705,soil,Dan Ingemann Jensen,55.905585,12.362367,75.0,Deciduous woodland,2238504270-171705.JPG
1,2238511676,2017-09-17T00:00:00,2017.0,9.0,17.0,DK,Store Hareskov,17238.0,Mycena haematopus (Pers.) P.Kumm.,Fungi,...,DNK.1.12_1,Furesø,2238511676-246956,dead wood (including bark),Tarquin Netherway,55.769675,12.401108,15.0,Deciduous woodland,2238511676-246956.JPG
2,2883271536,2020-10-18T00:00:00,2020.0,10.0,18.0,DK,Lillerød,17289.0,Mycena rosea (Schumach.) Gramberg,Fungi,...,DNK.1.2_1,Allerød,2883271536-291057,soil,Dan Ingemann Jensen,55.878946,12.359130,40.0,Deciduous woodland,2883271536-291057.JPG
3,2446759186,2019-10-26T00:00:00,2019.0,10.0,26.0,DK,Ollerup,10079.0,Agaricus impudicus (Rea) Pilát,Fungi,...,DNK.5.18_1,Svendborg,2446759186-271681,soil,Dan Schou,55.068441,10.515358,50.0,Unmanaged deciduous woodland,2446759186-271681.JPG
4,2382321410,2019-08-30T00:00:00,2019.0,8.0,30.0,DK,"Dyrehave, Valdemars Slot (inkl Pederskov)",19939.0,Russula chloroides (Krombh.) Bres.,Fungi,...,DNK.5.18_1,Svendborg,2382321410-191132,soil,Kirsten Bjørnsson,55.013310,10.644585,75.0,Deciduous woodland,2382321410-191132.JPG


In [6]:
print(f"Unique species: {test_df['scientificName'].nunique()}")

Унікальних видів: 182


In [7]:
print(test_df['scientificName'].unique()[:10])

['Russula nigricans (Bull.) Fr.' 'Mycena haematopus (Pers.) P.Kumm.'
 'Mycena rosea (Schumach.) Gramberg' 'Agaricus impudicus (Rea) Pilát'
 'Russula chloroides (Krombh.) Bres.' 'Amanita rubescens (Pers.) Gray'
 'Russula vesca Fr.' 'Agaricus arvensis Schaeff.' 'Boletus edulis Bull.'
 'Russula violeipes Quél.']


In [8]:
#dict for poison/not poison mushrooms. 0 - not poison, 1 - poison
mushrooms_map = {
    'Russula cyanoxantha (Schaeff.) Fr.': 0, 'Russula ochroleuca (Pers.) Fr.': 0, 'Infundibulicybe gibba (Pers.) Harmaja': 0, 'Clitocybe nebularis (Batsch) Quél.': 0,
    'Agaricus essettei Bon': 0, 'Infundibulicybe costata (Kühner & Romagn.) Harmaja': 0, 'Agaricus campestris L.': 0, 'Agaricus augustus Fr.': 0,
    'Amanita submembranacea (Bon) Gröger': 0, 'Russula odorata Romagn.': 0, 'Russula faginea Romagn., 1967': 0, 'Amanita excelsa Gonn. & Rabenh.': 0,
    'Russula velenovskyi Melzer & Zvára': 0, 'Agaricus cupreobrunneus (Jul.Schäff. & Steer) Pilát': 0, 'Russula aurea Pers.': 0, 'Russula melliolens Quél.': 0,
    'Russula delica Fr.': 0, 'Agaricus langei (F.H.Møller & Jul.Schäff.) Maire': 0, 'Boletus aereus Secr.': 0, 'Russula ionochlora Romagn.': 0,
    'Amanita lividopallescens (Secr. ex Boud.) Kühner & Romagn.': 0, 'Agaricus litoralis (Wakef. & A.Pearson) Pilát': 0, 'Amanita strobiliformis Gonn. & Rabenh.': 0,
    'Russula nitida (Pers.) Fr.': 0, 'Russula velutipes Velen.': 0, 'Agaricus bisporus (J.E.Lange) Pilat': 0, 'Russula virescens (Schaeff.) Fr.': 0, 'Russula claroflava Grove':0,
    'Russula grisea (Batsch) Fr.': 0, 'Russula xerampelina (Schaeff.) Fr.': 0, 'Amanita vaginata (Bull.) Vittad.': 0, 'Amanita crocea (Quél.) Singer': 0, 'Russula depallens Fr.':0,
    'Agaricus subperonatus (J.E.Lange) Singer': 0, 'Russula integra (L.) Fr., 1838': 0, 'Agaricus crocodilinus Murrill': 0, 'Russula parazurea Jul.Schäff.': 0,
    'Agaricus bernardii (Quél.) Sacc.': 0, 'Amanita ceciliae (Berk. & Broome) Bas': 0, 'Russula cessans A.Pearson': 0, 'Agaricus sylvicola (Vittad.) Peck, 1872': 0,
    'Russula paludosa Britzelm.': 0, 'Xerocomus subtomentosus (L.) Fr.': 0, 'Agaricus bitorquis (Quél.) Sacc.': 0, 'Russula olivacea (Schaeff.) Fr.': 0, 'Russula adusta (Pers.) Fr.': 0,
    'Infundibulicybe squamulosa (Pers.) Harmaja': 0, 'Agaricus dulcidulus Schulzer': 0, 'Russula graveolens Romell': 0, 'Russula risigallina (Batsch) Sacc.': 0,
    'Russula aeruginea Fr.': 0, 'Clitocybe odora (Bull.) P.Kumm.': 0, 'Russula heterophylla (Fr.) Fr.': 0, 'Russula puellaris Fr.': 0, 'Russula romellii Maire': 0,
    'Boletus pinophilus Pilát & Dermek': 0, 'Russula seperina Dupain': 0, 'Russula brunneoviolacea Crawshay': 0, 'Russula laeta Jul.Schäff.': 0, 'Russula decolorans (Fr.) Fr.': 0,
    'Agaricus bohusii Bon': 0, 'Agaricus subfloccosus (J.E.Lange) Pilát': 0, 'Russula curtipes F.H.Møller & Jul.Schäff.': 0, 'Russula caerulea (Pers.) Fr.': 0,
    'Russula subrubens (J.E.Lange) Bon': 0, 'Agaricus brunneolus (J.E.Lange) Pilát': 0, 'Russula faustiana Sarnari': 0, 'Agaricus lanipes (F.H.Møller & Jul.Schäff.) Singer': 0
}

In [9]:
def mushroom_category(name):
  return mushrooms_map.get(name, 1)

In [10]:
test_df['mushroom_label'] = test_df['scientificName'].apply(mushroom_category)

In [11]:
print(test_df[['scientificName', 'mushroom_label']])

                             scientificName  mushroom_label
0             Russula nigricans (Bull.) Fr.               1
1         Mycena haematopus (Pers.) P.Kumm.               1
2         Mycena rosea (Schumach.) Gramberg               1
3            Agaricus impudicus (Rea) Pilát               1
4        Russula chloroides (Krombh.) Bres.               1
...                                     ...             ...
3635                   Boletus edulis Bull.               1
3636         Mycena pelianthina (Fr.) Quél.               1
3637  Clitocybe phyllophila (Pers.) P.Kumm.               1
3638           Amanita fulva (Schaeff.) Fr.               1
3639   Mycena galopus (Pers.) P.Kumm., 1871               1

[3640 rows x 2 columns]


In [12]:
test_df.sample(10)

,gbifID,eventDate,year,month,day,countryCode,locality,taxonID,scientificName,kingdom,...,level2Name,ImageUniqueID,Substrate,rightsHolder,Latitude,Longitude,CoorUncert,Habitat,image_path,mushroom_label
3358,2413148079,2019-09-10T00:00:00,2019.0,9.0,10.0,DK,Søgård,11772.0,Infundibulicybe gibba (Pers.) Harmaja,Fungi,...,Vejle,2413148079-117970,soil,Ken A. Alminde,55.671813,9.334409,25.0,Deciduous woodland,2413148079-117970.JPG,0
2548,2964219385,2020-11-04T00:00:00,2020.0,11.0,4.0,DK,Græsmark Skov,17218.0,Mycena diosma Krieglst. & Schwöbel,Fungi,...,Køge,2964219385-219352,leaf or needle litter,Benny T. Olsen,55.464891,12.091836,50.0,Deciduous woodland,2964219385-219352.JPG,1
3047,2862693409,2020-09-14T00:00:00,2020.0,9.0,14.0,DK,Rude Skov,11061.0,Boletus aereus Secr.,Fungi,...,Slagelse,2862693409-137193,soil,Yvonne Engmann,55.207431,11.489520,75.0,Unmanaged deciduous woodland,2862693409-137193.JPG,0
3361,2238572300,2018-10-09T00:00:00,2018.0,10.0,9.0,DK,Læsø,10252.0,"Amanita muscaria (L.) Lam., 1783",Fungi,...,Læsø,2238572300-184228,soil,Per Taudal Poulsen,57.271862,10.972707,25.0,Mixed woodland (with coniferous and deciduous ...,2238572300-184228.JPG,1
2494,2812994326,2020-07-08T00:00:00,2020.0,7.0,8.0,DK,Hjøllund,63479.0,Amanita rubescens (Pers.) Gray,Fungi,...,Silkeborg,2812994326-282161,soil,Frits Sørensen,56.077836,9.356931,15.0,Unmanaged coniferous woodland,2812994326-282161.JPG,1
1718,2427872033,2019-10-15T00:00:00,2019.0,10.0,15.0,DK,Øksenrade Skov,63482.0,Clitocybe nebularis (Batsch) Quél.,Fungi,...,Middelfart,2427872033-46889,soil,Jacob Lindhardt Palm,55.495035,9.711485,25.0,Unmanaged deciduous woodland,2427872033-46889.JPG,0
618,2864910304,2020-09-17T00:00:00,2020.0,9.0,17.0,DK,Haarby,11746.0,Clitocybe agrestis Harmaja,Fungi,...,Assens,2864910304-138181,soil,Kirsten Munch Pontoppidan,55.225308,10.134014,15.0,lawn,2864910304-138181.JPG,1
3421,2238460880,2009-12-08T00:00:00,2009.0,12.0,8.0,DK,Eriksholm,11787.0,Clitocybe metachroa (Fr.) P.Kumm.,Fungi,...,Holbæk,2238460880-165982,soil,Tom Smidth,55.684960,11.801380,25.0,NaN,2238460880-165982.JPG,1
1037,2986455350,2020-11-25T00:00:00,2020.0,11.0,25.0,DK,Rantzausminde,63482.0,Clitocybe nebularis (Batsch) Quél.,Fungi,...,Svendborg,2986455350-221615,soil,Dan Schou,55.040664,10.551098,25.0,Mixed woodland (with coniferous and deciduous ...,2986455350-221615.JPG,0
2043,2237858476,2019-03-14T00:00:00,2019.0,3.0,14.0,DK,Jægerspris Slot,64062.0,Mycena tenerrima (Berk.) Sacc.,Fungi,...,Frederikssund,2237858476-224083,dead wood (including bark),Rasmus Riis-Hansen,55.851517,11.961568,25.0,Unmanaged deciduous woodland,2237858476-224083.JPG,1


In [12]:
import os
base_path = '/content/drive/MyDrive/Colab Notebooks/DF20M'
labels_path = os.path.join(base_path, 'labels')

In [15]:
def create_labels(df, output_dir):
    count = 0
    for _, row in df.iterrows():
        # get file, example: '2237852116-222680.JPG')
        filename = row['image_path'].split('/')[-1]
        name_only = os.path.splitext(filename)[0]

        label = row['mushroom_label']
        txt_path = os.path.join(output_dir, f"{name_only}.txt")

        # YOLO format: class x_center y_center width height
        with open(txt_path, 'w') as f:
            f.write(f"{label} 0.5 0.5 0.9 0.9")
        count += 1
    return count

In [16]:
labels_path = '/content/drive/MyDrive/labels/'
total_files = create_labels(test_df, labels_path)
print(f"Успішно створено {total_files} файлів розмітки у папці labels!")

Успішно створено 3640 файлів розмітки у папці labels!


In [14]:
mushrooms_map['Agaricus arvensis Schaeff.']

0

In [13]:
# clone repo YOLOv5
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
# install requirements
!pip install -r requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17822, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 17822 (delta 16), reused 6 (delta 6), pack-reused 17794 (from 2)
Receiving objects: 100% (17822/17822), 16.96 MiB | 23.89 MiB/s, done.
Resolving deltas: 100% (12144/12144), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 17.7 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [15]:
yaml_content = f"""
path: /content/drive/MyDrive/labels
train: images/train
val: images/val

names:
  0: edible
  1: poisonous
"""

with open('/content/drive/MyDrive/data.yaml', 'w') as f:
    f.write(yaml_content)

print("Файл data.yaml успішно створено на твоєму Диску!")

Файл data.yaml успішно створено на твоєму Диску!


In [16]:
import yaml

# making configuration dict
data_config = {
    'path': '/content/drive/MyDrive',
    'train': 'Colab Notebooks/DF20M', # path for images
    'val': 'Colab Notebooks/DF20M',
    'nc': 2,
    'names': ['edible', 'poisonous']
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_config, f)

In [32]:
import os
import shutil

# local structure
!mkdir -p /content/dataset/images
!mkdir -p /content/dataset/labels

# 1. copy labels
print("Копіюю лейбли...")
src_labels = '/content/drive/MyDrive/labels'
dst_labels = '/content/dataset/labels'
for f in os.listdir(src_labels):
    if f.endswith('.txt'):
        shutil.copy(os.path.join(src_labels, f), dst_labels)

# 2. make links for images
print("Створюю посилання на картинки...")
src_imgs = '/content/drive/MyDrive/Colab Notebooks/DF20M'
dst_imgs = '/content/dataset/images'

# make link for folder
!ln -s "{src_imgs}" /content/dataset/images_link

# get images names and make only for them links
labeled_names = [os.path.splitext(f)[0] for f in os.listdir(dst_labels)]
for name in labeled_names:
    for ext in ['.JPG', '.jpg']:
        img_path = os.path.join(src_imgs, name + ext)
        if os.path.exists(img_path):
            os.symlink(img_path, os.path.join('/content/dataset/images', name + ext))
            break

print(f"Готово! Знайдено лейблів: {len(os.listdir('/content/dataset/labels'))}")
print(f"Знайдено посилань на фото: {len(os.listdir('/content/dataset/images'))}")

Копіюю лейбли...
Створюю посилання на картинки...
Готово! Знайдено лейблів: 3640
Знайдено посилань на фото: 3640


In [33]:
import yaml

data_config = {
    'path': '/content/dataset',
    'train': 'images',
    'val': 'images',
    'nc': 2,
    'names': ['edible', 'poisonous']
}

with open('/content/yolov5/data/mushroom.yaml', 'w') as f:
    yaml.dump(data_config, f)

In [34]:
%cd /content/yolov5
!python train.py --img 640 --batch 32 --epochs 25 --data data/mushroom.yaml --weights yolov5s.pt

Выходные данные были обрезаны до нескольких последних строк (5000).
  with torch.cuda.amp.autocast(amp):
       3/24      11.7G    0.01832   0.009551   0.006804         91        640:  61% 69/114 [02:46<02:01,  2.71s/it]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       3/24      11.7G    0.01852   0.009563   0.006774         99        640:  61% 70/114 [02:48<01:58,  2.70s/it]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       3/24      11.7G    0.01846   0.009568   0.006815         99        640:  62% 71/114 [02:50<01:43,  2.40s/it]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda

In [35]:
import shutil
# best model
shutil.copy('/content/yolov5/runs/train/exp4/weights/best.pt', '/content/drive/MyDrive/mushroom_model_best.pt')
print("successfully saved to Google Drive!")

successfully saved to Google Drive!


In [38]:
!python detect.py --weights runs/train/exp4/weights/best.pt --img 640 --conf 0.25 --source гриб.jpg

detect: weights=['runs/train/exp4/weights/best.pt'], source=гриб.jpg, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 /content/yolov5/гриб.jpg: 640x512 1 poisonous, 46.9ms
Speed: 0.6ms pre-process, 46.9ms inference, 197.2ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/exp3


In [39]:
!python detect.py --weights runs/train/exp4/weights/best.pt --img 640 --conf 0.25 --source гриб2.jpg

detect: weights=['runs/train/exp4/weights/best.pt'], source=гриб2.jpg, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 /content/yolov5/гриб2.jpg: 640x640 1 poisonous, 10.3ms
Speed: 0.6ms pre-process, 10.3ms inference, 130.3ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/exp4


In [41]:
!python detect.py --weights runs/train/exp4/weights/best.pt --img 640 --conf 0.25 --source гриб3.jpg

detect: weights=['runs/train/exp4/weights/best.pt'], source=гриб3.jpg, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 /content/yolov5/гриб3.jpg: 480x640 1 poisonous, 50.8ms
Speed: 0.7ms pre-process, 50.8ms inference, 168.9ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/exp6


In [43]:
!python detect.py --weights runs/train/exp4/weights/best.pt --img 640 --conf 0.25 --source гриб4.jpg

detect: weights=['runs/train/exp4/weights/best.pt'], source=гриб4.jpg, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 /content/yolov5/гриб4.jpg: 448x640 1 poisonous, 50.6ms
Speed: 0.8ms pre-process, 50.6ms inference, 138.6ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/exp8


In [44]:
!python detect.py --weights runs/train/exp4/weights/best.pt --img 640 --conf 0.75 --source гриб4.jpg

detect: weights=['runs/train/exp4/weights/best.pt'], source=гриб4.jpg, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.75, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 /content/yolov5/гриб4.jpg: 448x640 (no detections), 31.9ms
Speed: 0.6ms pre-process, 31.9ms inference, 30.2ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/exp9


In [45]:
!python detect.py --weights runs/train/exp4/weights/best.pt --img 640 --conf 0.25 --source гриб4.jpg

detect: weights=['runs/train/exp4/weights/best.pt'], source=гриб4.jpg, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 /content/yolov5/гриб4.jpg: 448x640 1 poisonous, 39.0ms
Speed: 0.7ms pre-process, 39.0ms inference, 109.7ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/exp10


In [46]:
import shutil
from google.colab import files


shutil.make_archive('mushroom_results', 'zip', '/content/yolov5/runs/train/exp4')

files.download('mushroom_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>